# Experiment 17 — P1 Injection Recovery Calibration

Gate 2: verify axis recovery before any holdout claim. See `docs/studies/P1_CMB_RADON_SCAR_PREREG.md`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "polomni").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
from polomni.observatory.ingest.healpix_loader import synthetic_cmb_map
from polomni.observatory.scoring.rble_signature import compute_rble_signature, inject_synthetic_scar


def axis_error_deg(true_axis, recovered):
    a = np.asarray(recovered, float)
    a /= np.linalg.norm(a) + 1e-15
    t = np.asarray(true_axis, float)
    t /= np.linalg.norm(t) + 1e-15
    return np.degrees(np.arccos(np.clip(abs(np.dot(a, t)), -1, 1)))

nside = 32
snrs = {1: 2.0, 2: 4.0, 3: 8.0, 5: 12.0}
results = {}
for snr, amp in snrs.items():
    errs = []
    for trial in range(30):
        axis = np.random.randn(3)
        axis /= np.linalg.norm(axis)
        cmb = synthetic_cmb_map(nside, seed=trial)
        scarred = inject_synthetic_scar(cmb, axis, amplitude=amp)
        rep = compute_rble_signature(scarred, scan_angles=24)
        errs.append(axis_error_deg(axis, rep.preferred_axis))
    results[snr] = errs
    print(f"SNR={snr} median err={np.median(errs):.1f}° pass={(np.array(errs)<5).mean():.0%}")

plt.figure(figsize=(8,4))
plt.boxplot([results[k] for k in sorted(results)], labels=[f"SNR {k}" for k in sorted(results)])
plt.axhline(5, color="r", ls="--", label="5° threshold")
plt.ylabel("Axis error (deg)")
plt.legend()
plt.title("P1 injection recovery")
plt.show()